# Burla GPU vector-embedding demo

Embed a slice of English Wikipedia (`wikimedia/wikipedia` `20231101.en`) on Burla A100 workers with `BAAI/bge-large-en-v1.5`, then run a semantic-search query over the results.

### How it works

- **Stage 1 (CPU, parallel)** — each worker downloads one Wikipedia parquet shard from HuggingFace, takes the first `ARTICLES_PER_SHARD` rows, writes JSONL to `/workspace/shared`.
- **Stage 2 (GPU A100, parallel)** — each worker embeds its JSONL shard with the model baked into the image, writes `.npy` + `.json` outputs.
- **Stage 3 (GPU A100, 1 call)** — embed the query string.
- **Stage 4 (client)** — cosine-similarity top-K, print titles.

### Prerequisites

1. **Run this notebook kernel with Python 3.11.** Burla requires an exact `major.minor` match between client and worker, and the image ships Python 3.11.9.
2. `burla login` done.
3. A Burla cluster exists (started from the dashboard or grown by this script).
4. The image `jakezuliani/burla-embedder:latest` is on Docker Hub (built from the `Dockerfile` in this folder).
5. GCP A100 quota in the cluster's region.

See `README.md` for gotchas (package-sync clobbering CUDA, etc.).

## Setup

Top-level imports are intentionally minimal: **no** `torch`, **no** `datasets`, **no** `numpy`. If any of those appear here, Burla will detect them in `download_shard.__globals__` and reinstall the client's CPU wheels on top of the image's CUDA build, breaking the GPU path. ML libs live **inside** the worker functions.

In [ ]:
import json
from pathlib import Path
from burla import remote_parallel_map

IMAGE = "jakezuliani/burla-embedder:latest"
MODEL_NAME = "BAAI/bge-large-en-v1.5"
SHARED_ROOT = "/workspace/shared/vector_embeddings_demo"

N_PARQUET_FILES = 41
PARQUET_URL_TEMPLATE = (
    "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/"
    "20231101.en/train-{shard_idx:05d}-of-00041.parquet"
)

# Conservative notebook defaults. Bump TOTAL_ARTICLES / MAX_GPU_PARALLELISM
# for a bigger run. The full-scale demo in `demo.py` uses 50000 / 8.
TOTAL_ARTICLES = 5_000
ARTICLES_PER_SHARD = 1_000
N_SHARDS = TOTAL_ARTICLES // ARTICLES_PER_SHARD
MAX_GPU_PARALLELISM = 4

QUERY = "Who invented the telephone?"
TOP_K = 5

print(f"Preparing to embed {TOTAL_ARTICLES} articles across {N_SHARDS} shards.")
print(f"Will use up to {MAX_GPU_PARALLELISM} A100s concurrently in Stage 2.")

## Stage 1 — Download shards on CPU workers

Each worker downloads one parquet shard directly from HuggingFace's CDN and writes JSONL to the shared filesystem.

In [ ]:
def download_shard(shard_idx, articles_per_shard, shared_root):
    import io
    import json
    import urllib.request
    from pathlib import Path
    import pyarrow.parquet as pq

    parquet_url = PARQUET_URL_TEMPLATE.format(shard_idx=shard_idx % N_PARQUET_FILES)
    req = urllib.request.Request(parquet_url, headers={"User-Agent": "burla-demo/1.0"})
    with urllib.request.urlopen(req) as response:
        parquet_bytes = response.read()
    table = pq.read_table(io.BytesIO(parquet_bytes))
    n = min(articles_per_shard, len(table))
    table = table.slice(0, n)

    out_path = Path(shared_root) / "texts" / f"shard-{shard_idx:05d}.jsonl"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w") as f:
        for row in table.to_pylist():
            record = {
                "id": row["id"],
                "title": row["title"],
                "text": (row.get("text") or "")[:2000],
            }
            f.write(json.dumps(record) + "\n")
    print(f"shard {shard_idx}: wrote {n} articles to {out_path}")
    return str(out_path)


download_inputs = [(i, ARTICLES_PER_SHARD, SHARED_ROOT) for i in range(N_SHARDS)]
text_paths = remote_parallel_map(
    download_shard,
    download_inputs,
    image=IMAGE,
    grow=True,
    func_cpu=2,
    func_ram=8,
    max_parallelism=N_SHARDS,
)
print(f"Stage 1 done: {len(text_paths)} JSONL shards written")

## Stage 2 — Embed on A100 GPUs

The model is baked into the image so there's no HuggingFace download at job start. `cache = {}` at module level keeps the model resident between calls on the same worker.

In [ ]:
cache = {}


def embed_shard(shard_path, model_name, shared_root):
    import json
    from pathlib import Path
    import numpy as np
    import torch
    from sentence_transformers import SentenceTransformer

    if "model" not in cache:
        print(f"loading {model_name} (cuda={torch.cuda.is_available()}) ...")
        cache["model"] = SentenceTransformer(model_name, device="cuda")
    model = cache["model"]

    ids, titles, texts = [], [], []
    for line in Path(shard_path).read_text().splitlines():
        row = json.loads(line)
        ids.append(row["id"])
        titles.append(row["title"])
        texts.append(f"{row['title']}\n\n{row['text']}")

    vecs = model.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype("float32")

    shard_name = Path(shard_path).stem.replace("shard-", "")
    emb_dir = Path(shared_root) / "embeddings"
    emb_dir.mkdir(parents=True, exist_ok=True)
    emb_path = emb_dir / f"emb-{shard_name}.npy"
    ids_path = emb_dir / f"ids-{shard_name}.json"
    np.save(emb_path, vecs)
    ids_path.write_text(json.dumps({"ids": ids, "titles": titles}))
    print(f"shard {shard_name}: wrote {len(ids)} vectors")
    return {"emb_path": str(emb_path), "ids_path": str(ids_path), "n": len(ids)}


embed_inputs = [(p, MODEL_NAME, SHARED_ROOT) for p in text_paths]
embed_results = remote_parallel_map(
    embed_shard,
    embed_inputs,
    image=IMAGE,
    grow=True,
    func_gpu="A100",
    max_parallelism=min(MAX_GPU_PARALLELISM, len(text_paths)),
)
total_vecs = sum(r["n"] for r in embed_results)
print(f"Stage 2 done: {total_vecs} vectors across {len(embed_results)} shards")

## Stage 3 — Embed the query string

Re-uses the same GPU worker path so the client never needs CUDA/torch locally.

In [ ]:
def embed_query(query, model_name):
    import torch
    from sentence_transformers import SentenceTransformer

    if "model" not in cache:
        print(f"loading {model_name} (cuda={torch.cuda.is_available()}) ...")
        cache["model"] = SentenceTransformer(model_name, device="cuda")
    model = cache["model"]
    vec = model.encode(
        [query],
        batch_size=1,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )[0].astype("float32")
    return vec.tolist()


query_vec_list = remote_parallel_map(
    embed_query,
    [(QUERY, MODEL_NAME)],
    image=IMAGE,
    grow=True,
    func_gpu="A100",
    max_parallelism=1,
)
print(f"Query embedded, length={len(query_vec_list[0])}")

## Stage 4 — Local similarity search

Embeddings are normalized, so cosine similarity reduces to a single matrix-vector product.

In [ ]:
import io
import os
import numpy as np
import google.auth
from google.cloud import storage

# /workspace/shared is mounted only inside workers. Pull the .npy + .json
# shards via the GCS API on the client side.
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or google.auth.default()[1]
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(f"{PROJECT_ID}-burla-shared-workspace")


def _download(worker_path):
    blob_name = worker_path.replace("/workspace/shared/", "", 1)
    return bucket.blob(blob_name).download_as_bytes()


query_vec = np.asarray(query_vec_list[0], dtype="float32")
matrices, titles_flat = [], []
for result in sorted(embed_results, key=lambda r: r["emb_path"]):
    matrices.append(np.load(io.BytesIO(_download(result["emb_path"]))))
    titles_flat.extend(json.loads(_download(result["ids_path"]))["titles"])
matrix = np.concatenate(matrices, axis=0)
print(f"Loaded {matrix.shape[0]} vectors of dim {matrix.shape[1]}")

scores = matrix @ query_vec
ranked = np.argsort(-scores)
seen_titles = set()
print(f"\nTop {TOP_K} results for: {QUERY!r}\n")
printed = 0
for idx in ranked:
    title = titles_flat[idx]
    if title in seen_titles:
        continue
    seen_titles.add(title)
    printed += 1
    print(f"  {printed}. [{scores[idx]:.4f}] {title}")
    if printed >= TOP_K:
        break